# Methodologies and Rationale

**A meta-analysis on behalf of the UV&Me UCLA DGSOM Program**

by Matthew Mendoza, Santa Monica College 

Mentor: Cassidy Lee, UCLA PhD & Medical Student

## Table of Contents
1. Reason and Purpose for meta-analysis
    * 1.1 Abstract

2. Finding Literature and Filtering Studies
    * 2.1 Included Studies
    * 2.2 Prisma Flowchart

3. Math & Statistics
    * 3.1 Missing Data and Non-Normal Curves
    * 3.2 Standardizing Effect Size
    * 3.3 Accounting for Paired Data Variance
    * 3.4 Pooling the Data (REML Optimization)
    * 3.5 The Knapp-Hartung Adjustment
    * 3.6 Identifying heterogeneity ($I^2$)

4. Risk of Bias Assessment
    * 4.1 Traffic Light Plot
    * 4.2 The Summary Bar Plot

5. Results and Pooling

6. Conclusions and Limitations
    * 6.1 Conclusion
    * 6.2 Limitations and Procedural Deviations

7. How You Can Contribute

8. References

<br>

---

## 1. Reason and Purpose for meta-analysis:

**SPF** is often calculated under highly controlled and precise conditons, assuming perfect uniform application of **$2.0\text{ mg/cm}^2$**. Under real life conditions, human application is highly variable. The typical consumer only applies a fraction of the recommended amount (often between $0.5$ and $1.0\text{ mg/cm}^2$). This leaves consumers at high risk of the negative side-effects of UV radiation.

The purpose of this **meta-analysis** is to quantify exactly how much physical mass is applied on the skin with a second, consecutive layer of sunscreen. The goal is to determine whether **"double application"** is successful at making consumers apply the $2.0\text{ mg/cm}^2$ clinical standard. This meta-analysis also aims to standardize these findings across available scientific literature.




### 1.1 Abstract
This meta-analysis evaluates if a consecutive "double application" of sunscreen helps consumers reach the 2.0 mg/cm² clinical standard. We pooled four paired-design in vivo studies (n = 116) using REML optimization and Knapp-Hartung adjustments. While the initial model showed a statistically significant increase in applied mass, this result is statistically unstable. Because the eligible literature is so severely limited (k = 4), sensitivity analyses revealed that removing just one deviant study causes the confidence interval to cross zero. Therefore, while double application logically mitigates underapplication, the current body of evidence is simply too small and variable to state its exact efficacy with absolute statistical certainty.

<br>

---

<a id="section2"></a>
## 2. Finding Literature and Filtering Studies

Literature was found using Google Gemini's Deep Research, which yielded 11 possibly eligible papers. $N = 438$ These papers were then accessed through Santa Monica College's academic library access to Academic Search Complete and PubMed.

Links to all the papers can be found here:
- Link to google doc with all the papers and their citations.

Out of the 11 studies that were initially assessed, only four were deemed eligible. Aggregating data is only valid if the studies measure the exact same phenomenon, the following papers were excluded for their divergences:

* **Measuring Durability Over Time** *Bodekær (2008)* and *Ruvolo (2020)* were excluded because they evaluated the persistence of sunscreen over extended periods (6 - 8 hours). Rather than measuring the immediate physical mass of consecutive layers, they tracked efficacy after time with physical activity.

* **Variable Single Applications:** *Kim (2009)* applied single layers at artificially distinct thicknesses (e.g., $0.5, 1.0, 1.5\text{ mg/cm}^2$) rather than measuring a natural single application against a consecutive double application. However, their findings are still supportive of the conclusions of this study.

* **Layering with Cosmetics** *Kim (2021)* was excluded since it measured sunscreen layered with facial makeup. The chemical and physical differences led to the paper being removed from the statisitcal analysis.

* **Alternate Endpoints:** *Heerfordt (2018, Time)* evaluated *time spent* rubbing in the product rather than the physical mass applied. Additionally, this study used the same participants $n = 31$, thus pooling both studies would artificially inflate the numbers. *Pruim (1999)* measured SPF directly rather than quantifying physical application in $\text{mg/cm}^2$.

* **In Vitro Design:** *Fageon (2009)* was excluded because it utilized *in vitro* laboratory tests on polymethyl methacrylate plates (PMMA) instead of *in vivo* human skin. Prior to its exclusion, the Python pipeline was designed to handle studies where a smaller number indicated better results. A reflection of this fact can be found in the last CSV header *lower_is_better* and the *StudyMetadata* Class in *utilities.py*

&nbsp;

<a id="section2-1"></a>
### 2.1 Included Studies:

Following filtration of initial literature, the dataset was narrowed to $k = 4$ *in vivo* **paired (within-subject) design**.

* ***Heerfordt et al. (2018):*** *Why it was included:* This study measured the mass of consecutive applications across the whole body with ($N = 31$) participants using a topical fluorescent dye.
    * *Brief Limitation:* The data was only reported as medians and IQR, requiring statistical imputation to derive standard deviations

* ***Le Digabel et al. (2023):*** *Why it was included:* Using multispectral imaging, this study with ($N = 26$) participants captured the mass of a single application verses a consecutive reapplication 30 minutes later.
    * *Brief Limitation:* The measurement site was restricted exclusively to the face, which typically gets more attention than other parts of the body.

* ***Teramura et al. (2011):*** *Why it was included* This study with ($N = 23$) participants tracked application thickness for "high-performance" sunscreens, comparing single applications against double applications.
    * *Brief Limitation:* The interval between the first and second application was not mentioned, introducing slight ambiguity.

* ***De Villa et al. (2011):*** *Why it was included* This study with ($N = 36$) participants measured single versus double application weights 30 minutes apart on the forearms.
    * *Brief Limitation:* Dramatically lower baseline averages and highly skewed data variances compared to the rest of the studies due to researchers' "tape-stripping" and HPLC chemical extraction techniques for measuring.

<br>

<a id="section2-2"></a>
<h3 align="center">2.2 Prisma Flowchart</h3>

<p align="center">
<img src="figures/prisma_flowchart.png" width="650" alt="PRISMA Flowchart">
<br>
<i>Figure 1: PRISMA Flowchart</i>
</p>

<br>
<br>
<br>

---

<a id="section3"></a>
## 3. Math & Statistics

Raw clinical data cannot be averaged together. Different studies use different sample sizes, different measuring tools, and have different variables. The goal of this project transform disparate, **non-parametric** clinical data into a unified and continuous probabilisitc model capable of **inverse-variance pooling**.

<br>

### 3.1 Missing Data and Non-Normal Curves

When studies lack **standard deviation (SD)**, they are unable to be pooled. However, clinical trials often report **non-parametric** summaries (medians and ranges) because their underlying sample data is not normal. The best course of action is to **impute** the missing standard deviation and make an approximation of the normal curve.

> While imputing a parametric standard deviation from a non-parametric IQR forces potentially skewed data into a bell curve, this limitation is inherent to pooling continuous outcomes. To mitigate these shortcomings, all imputation strategies were paired with a strict **skewness evaluation** and a **sensitivity analysis** to monitor consistency and change.

**Wan et al. (2014) Method for imputing using IQR:**

*Heerfordt (2018)* reported data using means and **Interquartile Ranges (IQR)**, specifically a single application median of $0.83\text{ mg/cm}^2$ ($q_1 = 0.61$, $q_3 = 1.03$) and a double application median of $1.37\text{ mg/cm}^2$ ($q_1 = 1.01$, $q_3 = 1.87$).



In a **normal distribution**, the IQR spans exactly $1.349$ standard deviations. However, to account for sample size ($n$) in clinical trials, *Wan et al.* formalized the derivation of the standard deviation using the inverse cumulative normal distribution ($\Phi^{-1}$),

&nbsp;

$$SD \approx \frac{q_3 - q_1}{2 \times \Phi^{-1}\left(\frac{0.75n - 0.125}{n + 0.25}\right)}$$

&nbsp;

Where the inverse cumulative normal distribution ($\Phi^{-1}$) is defined as: 

<br>

$$\Phi(z) = P(Z \leq z) = \int_{-\infty}^{z} \frac{1}{\sqrt{2\pi}} e^{-\frac{t^2}{2}} dt$$

$$z = \Phi^{-1}(p) \iff \Phi(z) = p$$

<br>

When studies omit sample means, they are mathematically estimated from the reported medians and quartiles using established conversion formulas.

<br>

$$\bar{x} \approx \frac{q_1 + m + q_3}{3}$$


**Highly Skewed Data:**

To defend against artificially normalizing heavily skewed data, Python calculates **Bowley's coefficient of skewness** for every imputed study:

&nbsp;

$$\frac{|(q_3 - \text{median}) - (\text{median} - q_1)|}{q_3 - q_1}$$

&nbsp;

If this ratio exceeds the threshold of 0.20, the distribution is flagged as skewed. Applying this to *Heerfordt*'s single application ($q_1 = 0.61$, median $= 0.83$, $q_3 = 1.03$) yielded:

&nbsp;

$$\frac{|(1.03 - 0.83) - (0.83 - 0.61)|}{1.03 - 0.61} = \frac{|0.20 - 0.22|}{0.42} \approx 0.048$$

&nbsp;

With a coefficient of roughly 0.048 (well below the 0.20 threshold), mathematically the data is highly symmetrical.

Conversely, running this calculation with *De Villa (2011)*, who reported a median of $0.43$ and an IQR of $0.17$ to $1.07$:

&nbsp;

$$\frac{|(1.07 - 0.43) - (0.43 - 0.17)|}{1.07 - 0.17} = \frac{|0.64 - 0.26|}{0.90} \approx 0.422$$

&nbsp;

resulted in a large skewness ratio of $0.42$. Python flags the distribution as not being normal, triggering a **sensitivity analysis** to observe how the study changes overall heterogeneity.

**Hozo et al. (2005) Method for Imputing with Range Bounds**

When studies only report minimum and maximum observed weights, the dispersion of data heavily depends on **sample size** ($n$). As $n$ grows, the probability of capturing extreme, long-tail outliers increases, expanding the range. *Hozo et al.* accounts for this probabilistic expansion by dividing the range by a dynamic constant ($C$) that scales with $n$:

&nbsp;

$$SD \approx \frac{\text{max} - \text{min}}{C} \quad \text{where } C \in \{\sqrt{12}, 4, 6\}$$

&nbsp;

$$C =
\begin{cases}
\sqrt{12} & \text{if } n \le 15 \\
4 & \text{if } 15 < n \le 70 \\
6 & \text{if } n > 70
\end{cases}$$

&nbsp;

$C = \sqrt{12}$ (derived from the variance of a **continuous uniform distribution**)

$C = 4$ (encompassing roughly 95% of a **normal probability density function**)

$C = 6$ (encompassing 99.7% of the curve)

Similarly, missing sample means in these range-bound scenarios are estimated directly from the reported medians and minimum/maximum values.

<br>

$$\bar{x} \approx \frac{min + 2m + max}{4}$$

<br>

> **Note:** No studies actually fall under this category, however the Python script still includes it if any other studies are to be added in the future

**Imputing Variance via Coefficient of Variation (CV):**

In cases where a study reports their means but doesn't report variance of any kind (e.g., *Le Digabel 2023*), pooling isn't possible. To salvage the means, variance was imputed using a dynamic **Coefficient of Variation (CV)**.

Where CV is the ratio:

&nbsp;

$$CV = \frac{SD}{\text{Mean}}$$

&nbsp;

$$SD_{imputed} = \bar{x}_{i} \times \left( \frac{\sum_{j=1}^{k} n_j \frac{SD_j}{\bar{x}_j}}{\sum_{j=1}^{k} n_j} \right)$$

&nbsp;

* $SD_{imputed}$: The standard deviation we are trying to impute for the incomplete study

* $\bar{x}_{i}$: The mean reported by the incomplete study

* $n_j$: The sample size of the $j$-th study

* $\frac{SD_j}{\bar{x}_j}$: The individual Coefficient of Variation for other studies that did not report a standard deviation

&nbsp;

Python dynamically sums the CVs from all other reported studies to calculate a weighted global CV. This ratio is then multiplied against the study's mean to estimate a standard deviation when a mean is given but variance is not. Because of this layer of synthetic variance, a sensitivity analysis excluded CV-imputed studies, and the comparison against the other groups can be used to monitor artificial precision.

<br>

### 3.2 Standardizing Effect Size

*Le Digabel (2023)* used multispectral imaging (reporting mean applications of $1.04$ and $1.23\text{ mg/cm}^2$). *De Villa (2011)* used tape-stripping ($4$ x $4$ cm piece of adhesive tape, stuck to the skin and peeled off, repeated 5 times) and reported a lower median of $0.43$ and $0.95\text{ mg/cm}^2$

Because their raw starting baselines differed drastically, a **Standardized Mean Difference (SMD)** was used. Specifically **Cohen's $d$**.

&nbsp;


$$SD_{pooled} = \sqrt{\frac{(n_1 - 1)SD_1^2 + (n_2 - 1)SD_2^2}{n_1 + n_2 - 2}}$$


&nbsp;

$$d = \frac{M_2 - M_1}{SD_{pooled}}$$

&nbsp;

By dividing the difference in sunscreen mass ($M_2 - M_1$) by the pooled standard deviation, Cohen's $d$ acts as a universal, dimensionless scale. This means units of measurement are no longer considered, the standard deviation becomes the new unit of measurement.

**Hedges' $g$ Correction Factor:**

Cohen's $d$ assumes the sample's standard deviation is a perfect unbiased estimator of population variance. In small sample sizes, variance is mostly underestimated. This artificially inflates the effect size $d$. To correct this bias, we apply **Hedges' exact Gamma correct factor** ($J$).

&nbsp;

$$J = \frac{\Gamma(df/2)}{\sqrt{df/2} \times \Gamma((df-1)/2)}$$

&nbsp;

$$g = d \times J$$

&nbsp;

The **Gamma function** ($\Gamma$) is the extension of the factorial function. When we pass our degrees of freedom ($df$) through this ratio, $J$ yields a multiplier strictly less than $1.0$.

In the study *Teramura (2011)* with $n = 23$ volunteers, yielding $df = 22$, $g$ acts as a penalty. Mapping the biased Cohen's $d$ into the unbiased Hedges' $g$ mitigates bias.

<br>

### 3.3 Accounting for Paired Data Variance

This meta-analysis does not assume that Group 1 and Group 2 are independant populations. All included studies utilizied a **paired (within-subject) design**. The exact same volunteers were measured for one layer, and then measured again for layer two.

Treating this data as independent is invalid. Intuitively, if a volunteer applies a heavy first layer, chances are they might apply a heavy second layer. We account for this by integrating the **within-subject correlation coefficient** ($r$):


$$Var_d(r) = \frac{2(1-r)}{n} + \frac{d^2}{2(n-1)}$$

&nbsp;

$$SE_g(r) = \sqrt{Var_d(r) \times J^2}$$

&nbsp;

This formula computes the **Standard Error** for paired data in two distinct parts:

1. **$\frac{2(1-r)}{n}$**: The variance of the mean difference. As correlation ($r$) approaches 1.0, the term $(1-r)$ approaches zero. Strong correlation shrinks the variance because the paired within-subject differences are highly consistent.
2. **$\frac{d^2}{2(n-1)}$**: The statistical uncertainty inherent in estimating $d$. Crucially, the denominator relies strictly on $n-1$ degrees of freedom to respect the paired study design, rather than the $2n-2$ used for independent cohorts. This variance is subsequently scaled by $J^2$ to derive the final Standard Error for Hedges' $g$.
&nbsp;

**Imputing with the Correlation Set:**

While some studies provided p-values, they frequently derived from non-parametric tests (e.g., Wilcoxon signed-rank). Extracting an exact within-subject correlation coefficient ($r$) from a non-parametric p-value via a t-statistic conversion is invalid. Because the included papers did not report exact $r$ values, a sensitivity analysis was programmed to test a range of assumed coefficients:

&nbsp;

$$r \in \{0.2, 0.5, 0.8\}$$

&nbsp;

Visualizations and the primary model default to the moderate baseline:

&nbsp;

$$r_{baseline} = 0.5$$

&nbsp;

Testing across $r \in \{0.2, 0.5, 0.8\}$ showed consistency. Across all three values of r, the primary pooled effect size barely changed (Random Effects Mean ranging from $0.82$ to $0.83$). This confirms that the choice of the baseline $r = 0.5$ correlation does not artificially inflate or alter the fundamental findings of the primary model:

&nbsp;

$$\text{IN\_VIVO} \mid \text{MG/CM}^2 \ (k = 4) \to \text{RE Mean: } 0.82 \ [95\% \text{ CI: } 0.04, 1.61], \ I^2: 78.0\% \ [95\% \text{ CI: } 40.6\%, 91.9\%]$$

&nbsp;

> **Note:** This project is equipped to handle independent cohorts; it must be listed in the CSV and it will be grouped as necessary.



<a id="section3-4"></a>
### 3.4 Pooling the Data (REML Optimization)

Pooling is done with **Inverse Variance weighting** ($W \propto 1/V$). Studies with higher precision receive more weight. A **Random-Effects model** was utilized, assuming a universal sunscreen application weight does not exist due to differences in wait times, and other confounding variables. This introduces **$\tau^2$ (Tau-squared)**, the measure of how the results of the studies vary from one another, in standard deviations.

**Restricted Maximum Likelihood (REML)** is used via scalar optimization. REML maximizes a likelihood function calculated from a transformed set of data, effectively isolating the variance components and yielding an estimate of $\tau^2$. In other words, the computer takes our approximate parametric data that we imputed, and guesses numbers for $\tau^2$ until it finds a maximum likelihood $\tau^2$. A maximum likelihood represents the probability that this specific $\tau^2$ value is the best explanation for that data.

&nbsp;

$$W_i = \frac{1}{SE_i^2 + \tau^2}$$

&nbsp;

By adding $\tau^2$ to the denominator, the model flattens the weights. This prevents a low-variance study from disturbing the pooled estimate. This gives the meta-analysis a conservative approach by making it *harder* to get perfect results because this variable accounts for the real-world differences between researchers.

<a id="section3-5"></a>
### 3.5 The Knapp-Hartung Adjustment

Once pooled, a **95% Confidence Interval** must be established. A standard normal distribution assumes $\tau^2$ is the absolute, true population variance. With only $k = 4$ studies, $\tau^2$ is an uncertain estimate. Relying on a normal distribution will yield the *Type I* error to artificially inflate. To model this uncertainty, the **Knapp-Hartung adjustment** is used.

&nbsp;

$$q = \frac{1}{k-1} \sum_{i=1}^{k} w_i^* (g_i - \hat{g})^2$$

&nbsp;

* $k$: number of studies ($k = 4$)
* $g_i$: effect size of an individual study
* $\hat{g}$: pooled effect size (mean)
* $w_i^*$: weight assigned to each study

&nbsp;

To prevent the random-effects variance from artificially dropping below the fixed-effect variance in scenarios with extremely low heterogeneity, a **modified Knapp-Hartung adjustment** is used. This modification enforces a lower bound of $1.0$ on the $q$ multiplier:

&nbsp;

$$q^* = \max(1, q)$$

&nbsp;

Then, the final adjusted Standard Error ($SE_{mKH}$) is:

&nbsp;

$$SE_{mKH} = \sqrt{q^* \times Var_{pooled}}$$

&nbsp;

*(Where the standard pooled variance is the inverse of the sum of the REML weights)*:

&nbsp;

$$Var_{pooled} = \frac{1}{\sum_{i=1}^{k} W_i}$$

&nbsp;

This shifts the variance mapping to a **$t$-distribution**. Since the $t$-distribution features wider tails at low degrees of freedom, the critical multiplier expands beyond $1.96$. This generates a more conservative confidence interval that accounts for the small size of a $k = 4$ dataset.

<br>

<p align="center">
<img src="figures/distribution.jpeg" width="350" alt="Z vs T Distribution">
<br>
<i>Figure 2: Z vs T Distribution</i>
</p>

<br>

<br>

### 3.6 Identifying heterogeneity ($I^2$)

We quantify the degree of mathematical similarity among the studies by calculating **Cochran's $Q$** and converting it into $I^2$:

&nbsp;

$$Q = \sum_{i=1}^{k} W_i (g_i - \hat{g})^2$$

&nbsp;

$$I^2 = \left( \frac{Q - df}{Q} \right) \times 100\%$$

&nbsp;

 **Heterogeneity and Sensitivity Analysis**

The $I^2$ statistic represents the proportion of total variation across studies caused by true heterogeneity rather than random chance. Initial baseline analysis ($r = 0.5$) indicated high heterogeneity at $I^2 = 78.0\%$ ($95\%$ CI: $40.6\%$, $91.9\%$). 

Sensitivity analyses identified the primary drivers of this variance. Excluding the skewed *De Villa (2011)* dataset reduced heterogeneity to $I^2 = 2.5\%$ ($95\%$ CI: $0.0\%$, $89.9\%$). Separately, removing the variance-imputed *Le Digabel (2023)* dataset lowered it to $I^2 = 13.1\%$ ($95\%$ CI: $0.0\%$, $91.0\%$). These results suggest that the initial high heterogeneity was driven by methodological outliers rather than inconsistent effects across the included studies.

<br>
<br>

---

<br>

### 3.7 Confidence interval for heterogenity

Finally, Python will calculate the $95$% confidence interval for the $I^2$ statistic. First, the variance is mapped to the **$H$**-**statistic**. It represents the excess of Cochran's $Q$ over its degrees of freedom ($df = k - 1$).

&nbsp;

$$H = \sqrt{\frac{Q}{k-1}}$$

&nbsp;

(If $Q \leq k - 1$, then $H$ is bounded to $1.0$)

&nbsp;

Since $H$ is not normal, it's transformed into its natural logarithm ($ln(H)$) to calculate the standard error. The standard error or $ln(H)$ depends on the value of Cochran's $Q$ relative to the number of studies ($k$).

&nbsp;

If $Q > k$:$$SE(\ln H) = \frac{\ln(Q) - \ln(k-1)}{2 \left( \sqrt{2Q} - \sqrt{2k - 3} \right)}$$

&nbsp;

If $Q \leq k$:$$SE(\ln H) = \sqrt{\frac{1}{2(k-2)} \left( 1 - \frac{1}{3(k-2)^2} \right)}$$

&nbsp;

Now that the standard error is defined, the 95% confidence bounds for the logarithmetic statistic are calculated using the normal distribution ($Z = 1.96$):

&nbsp;

$$\ln H_{bounds} = \ln H \pm 1.96 \times SE(\ln H)$$

&nbsp;

These upper and lower bounds are subject to the inverse logarithm to get the original scale

&nbsp;

$$H_{bounds} = e^{\ln H_{bounds}}$$

&nbsp;

and is converted directly into the $I^2$ percentage bounds: 

&nbsp;

$$I^2_{bounds} = \left( \frac{H_{bounds}^2 - 1}{H_{bounds}^2} \right) \times 100\%$$

<br>

___









<a id="section4"></a>
## 4. Risk of Bias Assessment

Before interpreting the pooled results, the integrity of the clinical trials must be studied. The dataset was evaluated against the **Cochrane Risk of Bias 2 (RoB 2)** tool for randomized and paired trials.

<br>

<a id="section4-1"></a>
<h3 align="center">4.1 Traffic Light Plot</h3>


<p align="center">
<img src="figures/rob_traffic_light.png" width="600" alt="RoB Traffic Light Plot">
<br>
<i>Figure 3: RoB Traffic Light Plot</i>
</p>

<br>

* **Domain 1 (Randomization Process):** While all trials utilize a paired, within-subject design, the sequential application of consecutive sunscreen layers cannot be randomized in full-body interventions. Consequently, *Teramura*, *Heerfordt*, and *Le Digabel* present "Some concerns" for selection bias. *De Villa* successfully randomized the selection of the forearm for the second application, resulting in a Low risk of bias.

* **Domain 2 (Deviations from Intended Interventions):** It is practically impossible to blind participants to the intervention when they are applying a consecutive layer of sunscreen themselves. Participant awareness of the testing parameters introduces "Some concerns" for performance bias across all included studies.

* **Domain 3 (Missing Outcome Data):** All four studies reported zero attrition ($attrition\_n = 0$), resulting in a strictly Low Risk of bias for missing data.

* **Domain 4 (Measurement of Outcome):** *De Villa (2011)* utilized tape-stripping, a more variable extraction method compared to the techniques used in the other studies. As mentioned earlier, tape stripping was a $4$ x $4$ cm piece of adhesive tape, stuck to the skin and peeled off, repeated 5 times.

* **Domain 5 (Selection of the Reported Result):** All four included studies reported on their intended primary outcomes (applied sunscreen mass/thickness) without evidence of selectively withholding data. Consequently, the risk of selective reporting bias is Low.
&nbsp;

<a id="section4-2"></a>
<h3 align="center">4.2 The Summary Bar Plot</h3>


<p align="center">
<img src="figures/rob_summary_bar.png" width="800" alt="RoB Summary Bar Plot">
<br>
<i>Figure 4: RoB Summary Bar Plot</i>
</p>

<br>

The Summary Bar Plot aggregates these domain-level risks into a visual. A view of the overall health of the project.

Overall, the dataset presents a **moderate-to-low risk of bias**. The most significant concern is the inability to blind participants and researchers.

<br>
<br>

---

<a id="section5"></a>
## 5. Results and Pooling

Hedges' $g$ correction, paired standard errors, and REML variance weighting is visualized in the **Forest Plot** below.
It uses the $r_{baseline}$ of $0.5$

<p align="center">
<img src="figures/forest_plot_r05.png" width="800" alt="Forest Plot">
<br>
<i>Figure 5: Forest plot of the primary REML analysis.</i>
</p>

**Interpreting the Plot:**

* **Squares:** Each square represents the study's effect size (Hedges' $g$)

* **Square Size:** The size of each square corresponds directly to its REML random-effects weight ($1 / (SE^2 + \tau^2)$). *De Villa*, despite having a larger sample size $N = 36$, loses weight because of high variance.

* **Whiskers:** The horizontal lines extending from the squares represent the 95% Confidence Interval for that specific study.

* **The Diamond:** This represents the final, pooled global estimate. The center of the diamond sits at the calculated REML mean, and its width represents the Knapp-Hartung adjusted 95% Confidence Interval. Because the diamond does not cross $0.0$, the data shows a double application yields a statistical increase in applied physical mass ($mg/cm^2$)


&nbsp;

**Sensitivity Analyses:**
To validate the strength of the primary model ($r_{baseline} = 0.5$), two distinct sensitivity analyses were conducted to isolate potential shortcomings:

1. **Excluding Skewed IQR Data:** *De Villa (2011)* was excluded due to non-parametric skewness (Bowley's ratio: $0.42$). This successfully collapsed heterogeneity ($I^2 = 2.5\%$ [95% CI: 0.0%, 89.9%]), but the reduced sample size ($k = 3$) widened the confidence interval to $[-0.27, 2.23]$.

2. **Excluding CV-Imputed SDs:** *Le Digabel (2023)* was excluded to test the impact of synthetic variance imputation (the dynamic CV). This resulted in an $I^2$ of $13.1\%$ [95% CI: 0.0%, 91.0%] and widened the confidence interval to $[-0.37, 2.28]$.

&nbsp;

In both scenarios, because the adjusted confidence intervals cross zero, the statistical significance of the pooled effect is lost when stress-testing the dataset boundaries.

<br>
<br>

---

<a id="section6"></a>
## 6. Conclusions and Limitations

<br>

<a id="section6-1"></a>
### 6.1 Conclusion:

Primary analysis suggests a statistically significant increase in applied physical mass following a consecutive double application of sunscreen *in vivo*. However, this conclusion currently carries a low-to-moderate certainty. 

Sensitivity analyses stress-testing the model revealed that the statistical significance is highly sensitive to methodological exclusions. When either the highly skewed dataset (*De Villa 2011*) or the variance-imputed dataset (*Le Digabel 2023*) is excluded, the adjusted confidence intervals expand across zero. Therefore, while standard consumer behavior historically fails to achieve the $2.0\text{ mg/cm}^2$ standard, and double application likely increases overall mass, the exact magnitude and consistency of this effect remain too unstable across current literature to state with absolute statistical certainty.

Additionally, it can also be noted that when those same two studies are excluded, the confidence interval for the heterogenity statistic ($I^2$) has a lower bound of $0$, meaning that there is not enough data to be confident about *anything*, even the heterogeneity metric.

Thus, it can be reasonably concluded with caution that applying a two-layer consecutive application helps mitigate underapplication of sunscreen.

<br>

<a id="section6-2"></a>
### 6.2 Limitations

1. **Lack of Protocol Pre-Registration:** The protocol for this systematic review was not prospectively registered in a database such as PROSPERO.

2. **Single-Reviewer:** Literature screening, data extraction, and Risk of Bias assessments should be conducted by at least two independent reviewers, with a third mediating disputes. This project was conducted by a single undergraduate student, with a PhD mentor. This leaves the data highly vulnerable to human error and selection bias during the extraction phase.

3. **AI-Assisted Search and Database Bias:** The initial literature was conducted using AI-assisted deep research model to parse the web. Name's of the papers were then found and pasted into PubMed and Academic Search Complete. This approach bypasses search strings. This search did not include any other databases, nor did it attempt to access unpublished clinical trials. This might introduce potential search bias.

4. **Inability to Assess Publication Bias:** With a final dataset of only $k = 4$ studies, it is not possible to generate a reliable funnel plot or conduct an Egger's regression test. Thus, we cannot statistically rule out publication bias.

5. **Small Dataset and the Knapp-Hartung Penalty ($k = 4$):** While the initial literature search yielded 11 papers ($N = 438$), the filtration process only left $k = 4$ studies. The Knapp-Hartung adjustment heavily penalizes small sample sizes to prevent Type I errors. When removing either *De Villa (2011)* or *Le Digabel (2023)* in separate sensitivity analyses, the study count dropped to $k = 3$ ($df = 2$). This drop widens the tails of the $t$-distribution. Even though statistical heterogeneity improved in both scenarios, the penalty applied to the 95% Confidence Intervals expanded them to cross zero (e.g., $[-0.27, 2.23]$ and $[-0.37, 2.28]$). This highlights the instability of drawing definitive conclusions from limited literature.

6. **Clinical Heterogeneity in Wait Times:** The included studies lacked a standardized time interval between the first and second applications. *Le Digabel* and *De Villa* utilized a 30-minute interval, *Heerfordt* utilized 20 minutes, and *Teramura* did not specify.

7. **Industry Funding Bias:** Two of the included studies (*Le Digabel* and *Teramura*) were conducted by or received funding from cosmetic industry manufacturers.

<br>

---

## 7. How You Can Contribute

This project operates on a fully modular, object-oriented Python project designed for replicability. If you wish to add new clinical trials to this meta-analysis, you do not need to rewrite any mathematical logic.

If you are someone looking to adapt this code to your own needs, this project can do the following:

* **Automated Imputation**: Estimates any missing standard deviations depending on avaliable data
(using Wan et al., Hozo et al., or CV substitution techniques)

* **Standardization**: Converts raw data into Cohen's d and applies Hedges' correction factor g to mitigate small-sample bias

* **Flexibility**: Calculates variances for both paired (within-subject) and independant cohort trails

* **Visuals**: Generates forest plot, and other visuals

**How to add your data:**

1. Open the `data/extracted_papers.csv` file.

2. Add a new row following the exact column headers.

3. You do not need to calculate standard deviations or effect sizes manually. Simply input the raw data exactly as the study reported it (e.g., place the medians and IQRs in the `median`, `q1`, and `q3` columns, and only fill the `mean` and `sd` columns if explicitly mentioned in the paper).

4. Run `main.py`. The `formulas.py` and `analyzer.py` scripts will automatically detect the missing variance, trigger the appropriate Wan et al. or CV imputation matrices, recalculate the global REML weights, and automatically render new PDFs in the `/figures` directory.

5. If you have any questions, please email me @matthew7mendoza@gmail.com

<br>

---


<a id="section8"></a>
## 8. References

De Villa, D., Nagatomi, A. R., Paese, K., Guterres, S., & Cestari, T. F. (2011). Reapplication improves the amount of sunscreen, not its regularity, under real life conditions. *Photochemistry and Photobiology*, *87*(2), 457–460. [https://doi.org/10.1111/j.1751-1097.2010.00856.x](https://doi.org/10.1111/j.1751-1097.2010.00856.x)

<br>

Hedges, L. V. (1981). Distribution theory for Glass's estimator of effect size and related estimators. *Journal of Educational Statistics*, *6*(2), 107–128. [https://doi.org/10.3102/10769986006002107](https://doi.org/10.3102/10769986006002107)

<br>

Heerfordt, I. M., Torsnes, L. R., Philipsen, P. A., & Wulf, H. C. (2018). Sunscreen use optimized by two consecutive applications. *PLoS ONE*, *13*(3), e0193916. [https://doi.org/10.1371/journal.pone.0193916](https://doi.org/10.1371/journal.pone.0193916)

<br>

Higgins, J. P. T., Thomas, J., Chandler, J., Cumpston, M., Li, T., Page, M. J., & Welch, V. A. (Eds.). (2023). *Cochrane handbook for systematic reviews of interventions* (Version 6.4). Cochrane. [https://training.cochrane.org/handbook](https://training.cochrane.org/handbook)

<br>

Hozo, S. P., Djulbegovic, B., & Hozo, I. (2005). Estimating the mean and variance from the median, range, and the size of a sample. *BMC Medical Research Methodology*, *5*(1), 13. [https://doi.org/10.1186/1471-2288-5-13](https://doi.org/10.1186/1471-2288-5-13)

<br>

Knapp, G., & Hartung, J. (2003). Improved tests for a random effects meta-regression with a single covariate. *Statistics in Medicine*, *22*(17), 2693–2710. [https://doi.org/10.1002/sim.1482](https://doi.org/10.1002/sim.1482)

<br>

Le Digabel, J., Questel, E., Lauze, C., Carballido, F., & Josse, G. (2023). In vivo evaluation of sunscreen application by multispectral imaging: A new tool for educating sunscreen users. *Skin Research and Technology*, *29*(8), e13320. [https://doi.org/10.1111/srt.13320](https://doi.org/10.1111/srt.13320)

<br>

Teramura, T., Mizuno, M., Asano, H., Naito, N., Arakane, K., & Miyachi, Y. (2012). Relationship between sun-protection factor and application thickness in high-performance sunscreen: Double application of sunscreen is recommended. *Clinical and Experimental Dermatology*, *37*(8), 904–908. [https://doi.org/10.1111/j.1365-2230.2012.04388.x](https://doi.org/10.1111/j.1365-2230.2012.04388.x)

<br>

Wan, X., Wang, W., Liu, J., & Tong, T. (2014). Estimating the sample mean and standard deviation from the sample size, median, range and/or interquartile range. *BMC Medical Research Methodology*, *14*(1), 135. [https://doi.org/10.1186/1471-2288-14-135](https://doi.org/10.1186/1471-2288-14-135)